# Algoritmos voraces

Un algoritmo voraz construye la solución paso a paso eligiendo en cada momento la opción que parece mejor, sin volver atrás. Suele bastar con ordenar y recorrer, así que cuesta $O(n \log n)$, pero solo da el óptimo si el problema cumple la *propiedad de elección voraz*: hay una solución óptima que empieza por la elección voraz.

## 1. Cambio de monedas

Devolver una cantidad (en céntimos) con el menor número de monedas de euro cogiendo siempre la moneda más grande que quepa.

In [1]:
EUROS = [200, 100, 50, 20, 10, 5, 2, 1]

def cambio_voraz(cantidad, monedas=EUROS):
    resultado = []
    for moneda in sorted(monedas, reverse=True):
        veces, cantidad = divmod(cantidad, moneda)
        resultado += [moneda] * veces
    return resultado if cantidad == 0 else None

print(cambio_voraz(289))
print(cambio_voraz(1345))

[200, 50, 20, 10, 5, 2, 2]
[200, 200, 200, 200, 200, 200, 100, 20, 20, 5]


## 2. Cuando el cambio voraz falla

Con monedas `[1, 3, 4]` y cantidad 6 lo voraz da `4 + 1 + 1` (3 monedas), pero lo óptimo es `3 + 3`. Calcular el mínimo real con programación dinámica y comprobar que con las monedas de euro lo voraz siempre acierta.

In [2]:
def tabla_optimos(limite, monedas):
    minimo = [0] + [float("inf")] * limite
    for c in range(1, limite + 1):
        for m in monedas:
            if m <= c:
                minimo[c] = min(minimo[c], minimo[c - m] + 1)
    return minimo

print(cambio_voraz(6, [1, 3, 4]), "óptimo:", tabla_optimos(6, [1, 3, 4])[6], "monedas")

optimos = tabla_optimos(1000, EUROS)
assert all(len(cambio_voraz(c)) == optimos[c] for c in range(1001))
print("Con euros lo voraz es óptimo hasta 10 €")

[4, 1, 1] óptimo: 2 monedas
Con euros lo voraz es óptimo hasta 10 €


## 3. Selección de actividades

Dadas varias actividades con hora de inicio y fin, elegir el máximo número de ellas sin solapes en una misma sala. Voraz: ordenar por hora de fin y quedarse con cada actividad que empiece cuando ha terminado la última elegida. Comprobar contra fuerza bruta.

In [3]:
import random
from itertools import combinations

def selecciona_actividades(actividades):
    elegidas, fin_ultima = [], float("-inf")
    for inicio, fin in sorted(actividades, key=lambda a: a[1]):
        if inicio >= fin_ultima:
            elegidas.append((inicio, fin))
            fin_ultima = fin
    return elegidas

def compatibles(actividades):
    ordenadas = sorted(actividades)
    return all(a[1] <= b[0] for a, b in zip(ordenadas, ordenadas[1:]))

def fuerza_bruta_actividades(actividades):
    for r in range(len(actividades), 0, -1):
        if any(compatibles(c) for c in combinations(actividades, r)):
            return r
    return 0

def actividades_aleatorias(n):
    return [(i, i + random.randint(1, 6)) for i in (random.randint(0, 20) for _ in range(n))]

for _ in range(300):
    actividades = actividades_aleatorias(random.randint(0, 9))
    assert len(selecciona_actividades(actividades)) == fuerza_bruta_actividades(actividades)

agenda = [(9, 11), (10, 12), (11, 13), (8, 10), (12, 14), (13, 15), (14, 16), (9, 16)]
print(selecciona_actividades(agenda))

[(8, 10), (10, 12), (12, 14), (14, 16)]


## 4. Mochila fraccionaria

Si los objetos se pueden partir (especias a granel), lo óptimo es coger primero los de mayor valor por kilo y partir el último que no cabe entero. Comprobar que nunca da menos que la mochila 0/1 (sin partir) calculada por fuerza bruta.

In [4]:
def mochila_fraccionaria(objetos, capacidad):
    total, elegidos = 0, []
    for nombre, peso, valor in sorted(objetos, key=lambda o: o[2] / o[1], reverse=True):
        if capacidad == 0:
            break
        cantidad = min(peso, capacidad)
        total += valor * cantidad / peso
        capacidad -= cantidad
        elegidos.append((nombre, cantidad))
    return total, elegidos

def fuerza_bruta_mochila(objetos, capacidad):
    return max(sum(o[2] for o in c) for r in range(len(objetos) + 1)
               for c in combinations(objetos, r) if sum(o[1] for o in c) <= capacidad)

for _ in range(200):
    objetos = [(i, random.randint(1, 10), random.randint(1, 30)) for i in range(random.randint(1, 8))]
    capacidad = random.randint(0, 30)
    assert mochila_fraccionaria(objetos, capacidad)[0] >= fuerza_bruta_mochila(objetos, capacidad) - 1e-9

# (nombre, kilos, valor)
especias = [("azafrán", 1, 60), ("pimienta", 4, 40), ("comino", 5, 30), ("orégano", 3, 9), ("canela", 2, 16)]
print(mochila_fraccionaria(especias, 8))

(122.0, [('azafrán', 1), ('pimienta', 4), ('canela', 2), ('comino', 1)])


## 5. Planificación de tareas con plazos

Cada tarea dura una unidad de tiempo, tiene un plazo y un beneficio que solo se cobra si termina a tiempo. Voraz: tomar las tareas de mayor a menor beneficio y colocar cada una en el hueco libre más tardío que respete su plazo (si no hay, se descarta). Comprobar contra fuerza bruta.

In [5]:
def planifica(tareas):
    huecos = [None] * (max((plazo for _, plazo, _ in tareas), default=0) + 1)
    for nombre, plazo, beneficio in sorted(tareas, key=lambda t: t[2], reverse=True):
        for t in range(plazo, 0, -1):
            if huecos[t] is None:
                huecos[t] = (nombre, beneficio)
                break
    agenda = [h for h in huecos[1:] if h]
    return sum(b for _, b in agenda), [nombre for nombre, _ in agenda]

def factible(tareas):
    plazos = sorted(plazo for _, plazo, _ in tareas)
    return all(plazo >= k for k, plazo in enumerate(plazos, 1))

def fuerza_bruta_tareas(tareas):
    return max(sum(t[2] for t in c) for r in range(len(tareas) + 1)
               for c in combinations(tareas, r) if factible(c))

for _ in range(300):
    tareas = [(i, random.randint(1, 5), random.randint(1, 50)) for i in range(random.randint(0, 8))]
    assert planifica(tareas)[0] == fuerza_bruta_tareas(tareas)

# (nombre, plazo, beneficio)
tareas = [("informe", 2, 100), ("correo", 1, 19), ("reunión", 2, 27), ("factura", 1, 25), ("copia", 3, 15)]
print(planifica(tareas))

(142, ['reunión', 'informe', 'copia'])


## 6. Cubrir puntos con segmentos

Cubrir todos los puntos de una recta con el menor número de segmentos de longitud `L` (por ejemplo, farolas que alumbran `L` metros de una calle con casas). Voraz: ordenar los puntos y empezar cada segmento en el primer punto que aún no está cubierto. Comprobar contra fuerza bruta.

In [6]:
def cubre_puntos(puntos, longitud):
    segmentos = []
    for p in sorted(puntos):
        if not segmentos or p > segmentos[-1][1]:
            segmentos.append((p, p + longitud))
    return segmentos

def fuerza_bruta_segmentos(puntos, longitud):
    # basta probar segmentos que empiezan en un punto
    for k in range(len(puntos) + 1):
        for inicios in combinations(sorted(set(puntos)), k):
            if all(any(i <= p <= i + longitud for i in inicios) for p in puntos):
                return k

for _ in range(300):
    puntos = [random.randint(0, 30) for _ in range(random.randint(0, 8))]
    longitud = random.randint(0, 8)
    assert len(cubre_puntos(puntos, longitud)) == fuerza_bruta_segmentos(puntos, longitud)

casas = [1, 2, 3, 9, 10, 11, 12, 20, 27, 28]
print(cubre_puntos(casas, 4))

[(1, 5), (9, 13), (20, 24), (27, 31)]


## 7. Mínimo número de salas

Asignar todas las reuniones usando el menor número de salas. Voraz: recorrerlas por hora de inicio y reutilizar la sala que antes quede libre si ya lo está (un montículo con las horas de fin), o abrir una nueva. El resultado debe coincidir con el máximo número de reuniones que se solapan en un mismo instante.

In [7]:
import heapq

def asigna_salas(reuniones):
    libres = []  # montículo de (hora en que queda libre, sala)
    asignacion = []
    for inicio, fin in sorted(reuniones):
        if libres and libres[0][0] <= inicio:
            _, sala = heapq.heappop(libres)
        else:
            sala = len(libres) + 1
        heapq.heappush(libres, (fin, sala))
        asignacion.append((inicio, fin, sala))
    return len(libres), asignacion

def max_solapadas(reuniones):
    # en empate, el fin (-1) va antes que el inicio (+1)
    eventos = sorted([(inicio, 1) for inicio, _ in reuniones] + [(fin, -1) for _, fin in reuniones])
    actual = maximo = 0
    for _, cambio in eventos:
        actual += cambio
        maximo = max(maximo, actual)
    return maximo

for _ in range(500):
    reuniones = actividades_aleatorias(random.randint(0, 15))
    assert asigna_salas(reuniones)[0] == max_solapadas(reuniones)

salas, asignacion = asigna_salas(agenda)
print(salas, "salas")
for inicio, fin, sala in asignacion:
    print(f"{inicio:>2}-{fin:<2} -> sala {sala}")

3 salas
 8-10 -> sala 1
 9-11 -> sala 2
 9-16 -> sala 3
10-12 -> sala 1
11-13 -> sala 2
12-14 -> sala 1
13-15 -> sala 2
14-16 -> sala 1


## 8. Elegir bien el criterio voraz

En la selección de actividades hay otros criterios voraces que parecen razonables: empezar por la que antes empieza o por la más corta. Buscar contraejemplos con pruebas aleatorias contra la fuerza bruta para ver cuál falla.

In [8]:
criterios = {
    "termina antes": lambda a: a[1],
    "empieza antes": lambda a: a[0],
    "más corta": lambda a: a[1] - a[0],
}

def voraz_con_criterio(actividades, criterio):
    elegidas = []
    for a in sorted(actividades, key=criterio):
        if all(a[1] <= b[0] or b[1] <= a[0] for b in elegidas):
            elegidas.append(a)
    return elegidas

random.seed(4)
for nombre, criterio in criterios.items():
    contraejemplo = None
    for _ in range(3000):
        actividades = actividades_aleatorias(random.randint(2, 5))
        if len(voraz_con_criterio(actividades, criterio)) < fuerza_bruta_actividades(actividades):
            contraejemplo = actividades
            break
    if contraejemplo:
        print(f"{nombre}: falla con {contraejemplo}: elige {voraz_con_criterio(contraejemplo, criterio)}, "
              f"pero caben {fuerza_bruta_actividades(contraejemplo)}")
    else:
        print(f"{nombre}: sin contraejemplos")

termina antes: sin contraejemplos
empieza antes: falla con [(14, 17), (17, 22), (11, 12), (13, 19)]: elige [(11, 12), (13, 19)], pero caben 3
más corta: falla con [(18, 20), (17, 20), (19, 23), (16, 19)]: elige [(18, 20)], pero caben 2


## 9. ¿Cuándo funciona lo voraz?

Lo voraz es óptimo si se cumplen dos propiedades:

- **Elección voraz**: alguna solución óptima contiene la elección voraz. Se suele demostrar con un argumento de intercambio: si una solución óptima no la contiene, se cambia uno de sus elementos por la elección voraz sin empeorarla (en las actividades, cambiar la primera por la que antes termina).
- **Subestructura óptima**: tras la elección queda un problema más pequeño del mismo tipo.

Si fallan hay que recurrir a programación dinámica o a búsqueda. Comprobar qué sistemas de monedas son *canónicos* (lo voraz siempre es óptimo) buscando el menor contraejemplo (si existe, es menor que la suma de las dos monedas mayores) y ver que el criterio de la mochila fraccionaria no sirve para la mochila 0/1.

In [9]:
def menor_contraejemplo(monedas):
    limite = sum(sorted(monedas)[-2:])
    optimos = tabla_optimos(limite, monedas)
    for c in range(1, limite):
        voraz = cambio_voraz(c, monedas)
        if len(voraz) > optimos[c]:
            return c, voraz, optimos[c]
    return None

sistemas = {"euro": EUROS, "dólar": [1, 5, 10, 25], "potencias de 2": [1, 2, 4, 8, 16],
            "[1, 3, 4]": [1, 3, 4], "[1, 7, 10]": [1, 7, 10], "libra antigua": [1, 3, 6, 12, 24, 30]}
for nombre, monedas in sistemas.items():
    contraejemplo = menor_contraejemplo(monedas)
    if contraejemplo:
        c, voraz, optimo = contraejemplo
        print(f"{nombre:<15} no canónico: {c} -> voraz {voraz}, óptimo {optimo} monedas")
    else:
        print(f"{nombre:<15} canónico")

euro            canónico
dólar           canónico
potencias de 2  canónico
[1, 3, 4]       no canónico: 6 -> voraz [4, 1, 1], óptimo 2 monedas
[1, 7, 10]      no canónico: 14 -> voraz [10, 1, 1, 1, 1], óptimo 2 monedas
libra antigua   no canónico: 48 -> voraz [30, 12, 6], óptimo 2 monedas


In [10]:
def mochila_01_voraz(objetos, capacidad):
    total = 0
    for _, peso, valor in sorted(objetos, key=lambda o: o[2] / o[1], reverse=True):
        if peso <= capacidad:
            total += valor
            capacidad -= peso
    return total

objetos = [("a", 10, 60), ("b", 20, 100), ("c", 30, 120)]
print("0/1 voraz:", mochila_01_voraz(objetos, 50), "| 0/1 óptimo:", fuerza_bruta_mochila(objetos, 50),
      "| fraccionaria:", mochila_fraccionaria(objetos, 50)[0])

0/1 voraz: 160 | 0/1 óptimo: 220 | fraccionaria: 240.0
